# CSE 234 Programming Assignment 3: Speculative Decoding

## Setup

In [1]:
!pip install torch torchvision transformers accelerate

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 23.6 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 KB 51.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 KB 58.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 50.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.7/781.7 KB 47.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 KB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 KB 80.4 MB/s eta 0:00:00


In [1]:
import os
import torch
import time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Tuple, Dict, Optional

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!nvidia-smi

Thu Mar 20 17:29:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:06:00.0 Off |                    0 |
| N/A   53C    P0             53W /  400W |       1MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Speculative Decoding

In [3]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """
        Initialize the speculative decoder with target and draft models.

        Args:
            target_model_name: HuggingFace model ID for the larger target model.
            draft_model_name: HuggingFace model ID for the smaller draft model.
            device: Device to run models on ("cuda" or "cpu").
        """
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        # Ensure tokenizers are compatible
        assert self.target_tokenizer.vocab == self.draft_tokenizer.vocab, "Tokenizers must be compatible"

    def initialize_target_model(self, model_name: str):
        """Initialize the larger target model with caching enabled and proper pad token."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # TODO: Implement target model initialization
        # 1. Set the pad token if it doesn't exist
        # 2. Load the model with appropriate settings for inference
        # 3. Enable any optimizations that might help with performance
        
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16, 
            device_map=self.device
        ).eval()
        model.config.use_cache = True
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """
        Initialize a smaller, faster draft model with proper pad token.
        Uses lower precision and additional optimizations.
        """
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        # TODO: Implement draft model initialization
        # 1. Set the pad token if it doesn't exist
        # 2. Load the model with appropriate settings for inference
        # 3. Enable any optimizations that might help with performance
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map=self.device
        ).eval()
        
        model = torch.compile(model, mode="reduce-overhead", fullgraph=True)
        model.config.use_cache = True
        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate speculative tokens in one forward call using the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        # TODO: Implement draft token generation
        # 1. Use the draft model to generate tokens
        # 2. Extract only the new tokens (not including the input)
        # 3. Return the newly generated tokens
        with torch.no_grad():
            outputs = self.draft_model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                pad_token_id=self.draft_tokenizer.pad_token_id,
                do_sample=False,
                use_cache=True
            )
        new_tokens = outputs[:, -num_speculative_tokens:]
        return new_tokens

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Index of the first rejected token (if all accepted, equals draft_tokens.shape[1]).
        """
        # TODO: Implement efficient verification of draft tokens
        # 1. Run target model on input_ids concatenated with draft_tokens
        # 2. Extract the logits for positions where draft tokens would be predicted
        # 3. Compare target model predictions with draft tokens
        # 4. Determine how many consecutive tokens were accepted before first mismatch
        combined_input = torch.cat([input_ids, draft_tokens], dim=1)
        combined_mask = torch.ones_like(combined_input)
        
        with torch.no_grad():
            outputs = self.target_model(
                combined_input,
                attention_mask=combined_mask,
                use_cache=True 
            ).logits

        # Logits extracted looking at draft token preds
        draft_logits = outputs[:, input_ids.shape[1]-1:-1, :]
        draft_preds = draft_logits.argmax(dim=-1).squeeze(0)

        # Find matches and mismatches list
        correct = (draft_tokens == draft_preds)
        mismatches_indices = (~correct[0]).nonzero(as_tuple=True)[0]

        if mismatches_indices.numel() > 0:
            accepted_position = mismatches_indices[0].item()
            accepted_tokens = draft_tokens[0, :accepted_position].tolist()
        else:
            accepted_position = draft_tokens.shape[1]
            accepted_tokens = draft_tokens[0].tolist()
            
        return accepted_tokens, accepted_position

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 15) -> str:
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.

        Returns:
            Generated text.
        """
        # Tokenize prompt
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        # Initialize counters for performance tracking
        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        # TODO: Implement the core speculative decoding loop
        # 1. Generate draft tokens using the draft model
        # 2. Verify draft tokens using the target model
        # 3. Accept verified tokens and append to the sequence
        # 4. For rejected tokens or if all tokens are accepted, generate a new token with the target model
        # 5. Stop when max_tokens is reached or an EOS token is generated
        while total_tokens_generated - prompt_length < max_tokens:
            remaining = max_tokens - (total_tokens_generated - prompt_length)
            current_num_speculative = min(num_speculative_tokens, remaining)
            
            draft_tokens = self.generate_draft_tokens(
                input_ids, 
                attention_mask,
                current_num_speculative
            )
            total_draft_tokens_proposed += draft_tokens.shape[1]
            
            accepted_tokens, accepted_pos = self.verify_tokens_vectorized(
                input_ids,
                draft_tokens,
                attention_mask
            )
            total_draft_tokens_accepted += len(accepted_tokens)
            
            # Update input_ids and attention_mask
            if len(accepted_tokens) > 0:
                input_ids = torch.cat([
                    input_ids,
                    torch.tensor([accepted_tokens], device=self.device)
                ], dim=1)
                attention_mask = torch.cat([
                    attention_mask,
                    torch.ones((1, len(accepted_tokens)), device=self.device)
                ], dim=1)
                total_tokens_generated += len(accepted_tokens)
            
            # If all rejected or after accepted tokens, generate one token with target
            if accepted_pos < draft_tokens.shape[1] or len(accepted_tokens) == 0:
                outputs = self.target_model.generate(
                    input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=1,
                    pad_token_id=self.target_tokenizer.pad_token_id,
                    do_sample=False,
                    use_cache=True
                )
                new_token = outputs[:, -1:]
                input_ids = torch.cat([input_ids, new_token], dim=1)
                attention_mask = torch.cat([
                    attention_mask,
                    torch.ones((1, 1), device=self.device)
                ], dim=1)
                total_tokens_generated += 1
            
            # Check for EOS token
            if input_ids[0, -1] == self.target_tokenizer.eos_token_id:
                break

        # Calculate performance metrics
        elapsed_time = time.time() - start_time
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(self, prompt: str, max_tokens: int = 100,
                  num_runs: int = 3, compare_baseline: bool = True, num_speculative_tokens: int = 15 ) -> Dict:
        """
        Benchmark the speculative decoder against baseline decoding.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate.
            num_runs: Number of benchmark runs.
            compare_baseline: Whether to compare with baseline (non-speculative) decoding.

        Returns:
            Dictionary with benchmark results.
        """
        results = {
            "speculative": {"times": [], "tokens_per_second": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None
        }

        # Benchmark speculative decoding.
        for _ in range(num_runs):
            start_time = time.time()
            output = self.speculative_decode(prompt, max_tokens=max_tokens,num_speculative_tokens = num_speculative_tokens)
            elapsed = time.time() - start_time
            prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
            output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
            tps = output_tokens / elapsed
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(tps)

        # Benchmark baseline decoding.
        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                start_time = time.time()
                with torch.no_grad():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_length=input_ids.shape[1] + max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id
                    )
                elapsed = time.time() - start_time
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                tps = output_tokens / elapsed
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(tps)

        for method in results.keys():
            if results[method] is not None:
                avg_time = sum(results[method]["times"]) / num_runs
                avg_tps = sum(results[method]["tokens_per_second"]) / num_runs
                results[method]["avg_time"] = avg_time
                results[method]["avg_tokens_per_second"] = avg_tps

        if compare_baseline:
            speedup = results["baseline"]["avg_time"]/ results["speculative"]["avg_time"]
            results["speedup"] = speedup
            results["latency_reduction"] = (1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]) * 100
            # print(f"Speculative decoding speedup: {speedup:.2f}x")
            # print(f"Latency reduction: {results['latency_reduction']:.2f}%")

        return results

## Test

There are 2 blocks in Test section

-- One which have the original settings for benchmarking and getting the speedup and token acceptance rate. I got 1.3-1.4x and 85%+ acceptance rate. But there is no mention of that we can play with the parameters also. I asked on piazza and no one replied so I am putting second block.
-- Second code cell have the same cell but parametrized for best output. (close to 1.7x and 90%+ acceptance rate). Pls consider this for extra marks if you can, else above code cell untouched is also fine.

In [6]:
# Untouched test block giving ~ 1.3x speedup and 86% token acceptance rate..
target_model_name = "EleutherAI/pythia-1.4b-deduped"  # Larger target model
draft_model_name = "EleutherAI/pythia-160m-deduped"   # Smaller draft model


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped
Loading draft model: EleutherAI/pythia-160m-deduped

Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is
Generated 100 tokens in 1.08 seconds
Tokens per second: 92.58
Draft token acceptance rate: 86.84%
Generated 100 tokens in 1.08 seconds
Tokens per second: 92.89
Draft token acceptance rate: 86.84%
Average speculative decoding time: 1.08 seconds
Average speculative tokens per second: 92.63
Average baseline decoding time: 1.51 seconds
Average baseline tokens per second: 66.24
Speedup: 1.40x
Latency reduction: 28.49%

Benchmarking Prompt 2:
Prompt: Write a short story about a robot learning to feel emotions:
Generated 100 tokens in 1.17 seconds
Tokens per second: 85.82
Draft token acceptance rate: 80.99%
Generated 100 tokens in 1.17 seconds
Tokens per second: 85.68
Draft token acceptance rate: 80.99%
Generated 100 tokens in 1.17 seconds
Tokens per second: 85.73
Draft token acceptance rate: 80.99%
Average speculativ

In [28]:
# Parametrized test block giving ~ 1.7x speedup and 92% token acceptance rate..
target_model_name = "EleutherAI/pythia-1.4b-deduped"  # Larger target model
draft_model_name = "EleutherAI/pythia-160m-deduped"   # Smaller draft model


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=1000,
        num_runs=3,
        num_speculative_tokens=50,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped
Loading draft model: EleutherAI/pythia-160m-deduped

Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is
Generated 1000 tokens in 9.18 seconds
Tokens per second: 108.91
Draft token acceptance rate: 95.23%
Generated 1000 tokens in 9.13 seconds
Tokens per second: 109.48
Draft token acceptance rate: 95.23%
Average speculative decoding time: 9.15 seconds
Average speculative tokens per second: 109.29
Average baseline decoding time: 15.41 seconds
Average baseline tokens per second: 64.89
Speedup: 1.68x
Latency reduction: 40.63%

Benchmarking Prompt 2:
Prompt: Write a short story about a robot learning to feel emotions:
Generated 1000 tokens in 9.49 seconds
Tokens per second: 105.36
Draft token acceptance rate: 91.48%
Generated 1000 tokens in 9.50 seconds
Tokens per second: 105.31
Draft token acceptance rate: 91.48%
Average speculative decoding time: 9.50 seconds
Average speculative tokens per second: 105.23
Average base

## Different settings


In [7]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"  # Larger target model
draft_model_name = "EleutherAI/pythia-160m-deduped"   # Smaller draft model

# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

test_prompts = [
    "The future of Artificial Intelligence is", 
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'.", 
]
num_speculative_tokens_values = [50, 150, 550, 1500]
max_tokens_list = [500, 1000, 2000]
results_dict = {}

for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i + 1}:")
    print(f"Prompt: {prompt}")

    results_dict[i] = {}

    for num_speculative_tokens in num_speculative_tokens_values:
        results_dict[i][num_speculative_tokens] = {}

        for max_tokens in max_tokens_list:
            print(f"\n--- Testing with num_speculative_tokens = {num_speculative_tokens}, max_tokens = {max_tokens} ---")

            results = decoder.benchmark(
                prompt=prompt,
                max_tokens=max_tokens,
                num_runs=3,
                compare_baseline=True,
                num_speculative_tokens=num_speculative_tokens
            )

            print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
            print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

            if results["baseline"] is not None:
                print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
                print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
                print(f"Speedup: {results['speedup']:.2f}x")
                print(f"Latency reduction: {results['latency_reduction']:.2f}%")

            # Store results in the dictionary
            results_dict[i][num_speculative_tokens][max_tokens] = {
                "speedup": results.get("speedup", None),
                "draft_acceptance_rate": results.get("draft_acceptance_rate", None)
            }
            

Loading target model: EleutherAI/pythia-1.4b-deduped
Loading draft model: EleutherAI/pythia-160m-deduped

Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is

--- Testing with num_speculative_tokens = 50, max_tokens = 500 ---
Generated 500 tokens in 4.71 seconds
Tokens per second: 106.13
Draft token acceptance rate: 90.89%
Generated 500 tokens in 4.67 seconds
Tokens per second: 107.09
Draft token acceptance rate: 90.89%
Generated 500 tokens in 4.68 seconds
Tokens per second: 106.77
Draft token acceptance rate: 90.89%
Average speculative decoding time: 4.69 seconds
Average speculative tokens per second: 106.64
Average baseline decoding time: 7.62 seconds
Average baseline tokens per second: 65.65
Speedup: 1.62x
Latency reduction: 38.43%

--- Testing with num_speculative_tokens = 50, max_tokens = 1000 ---
Generated 1000 tokens in 9.06 seconds
Tokens per second: 110.37
Draft token acceptance rate: 95.23%
Generated 1000 tokens in 9.04 seconds
Tokens per second: 110.60
Dr

## Bonus

In [23]:
target_model_name = "EleutherAI/gpt-neo-1.3B" 
draft_model_name = "distilgpt2" 


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/gpt-neo-1.3B
Loading draft model: distilgpt2

Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is
Generated 100 tokens in 1.32 seconds
Tokens per second: 75.84
Draft token acceptance rate: 47.31%
Generated 100 tokens in 1.32 seconds
Tokens per second: 75.92
Draft token acceptance rate: 47.31%
Generated 100 tokens in 1.32 seconds
Tokens per second: 75.97
Draft token acceptance rate: 47.31%
Average speculative decoding time: 1.32 seconds
Average speculative tokens per second: 75.85
Average baseline decoding time: 1.92 seconds
Average baseline tokens per second: 52.07
Speedup: 1.46x
Latency reduction: 31.35%

Benchmarking Prompt 2:
Prompt: Write a short story about a robot learning to feel emotions:
Generated 100 tokens in 0.85 seconds
Tokens per second: 117.02
Draft token acceptance rate: 70.80%
Generated 100 tokens in 0.86 seconds
Tokens per second: 116.78
Draft token acceptance rate: 70.80%
Generated 100 tokens in 0.86 seconds
Tokens